In [ ]:
from google.colab import files

uploaded = files.upload()
pdf_path = list(uploaded.keys())[0]
print("Uploaded:", pdf_path)

In [ ]:
import pdfplumber

def get_all_lines(pdf_path):
    all_pages_lines = []

    with pdfplumber.open(pdf_path) as pdf:
        for page_number, page in enumerate(pdf.pages):
            words = page.extract_words()
            words.sort(key=lambda w: (w["top"], w["x0"]))

            lines = []
            current_line = []
            current_top = None
            tolerance = 3

            for w in words:
                if current_top is None or abs(w["top"] - current_top) <= tolerance:
                    current_line.append(w)
                    current_top = w["top"] if current_top is None else current_top
                else:
                    lines.append(current_line)
                    current_line = [w]
                    current_top = w["top"]
            if current_line:
                lines.append(current_line)

            page_lines = [" ".join(w["text"] for w in line) for line in lines]
            all_pages_lines.append(page_lines)

    return all_pages_lines

all_pages_lines = get_all_lines(pdf_path)
print("Extracted", sum(len(p) for p in all_pages_lines), "lines total")

In [ ]:
def get_all_tables(pdf_path):
    all_tables = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            tables = page.extract_tables()
            all_tables.append(tables)
    return all_tables

all_tables = get_all_tables(pdf_path)

In [ ]:
from docx import Document

def build_docx(pdf_path, all_pages_lines, all_tables, output_path):
    doc = Document()

    for page_number, page_lines in enumerate(all_pages_lines):
        doc.add_heading(f"Page {page_number + 1}", level=2)

        for line in page_lines:
            doc.add_paragraph(line)

        page_tables = all_tables[page_number] if page_number < len(all_tables) else []
        for table_data in page_tables:
            if not table_data:
                continue
            rows = len(table_data)
            cols = len(table_data[0])
            table = doc.add_table(rows=rows, cols=cols)
            table.style = "Table Grid"

            for r, row_data in enumerate(table_data):
                for c, cell_value in enumerate(row_data):
                    table.rows[r].cells[c].text = cell_value if cell_value else ""

        if page_number < len(all_pages_lines) - 1:
            doc.add_page_break()

    doc.save(output_path)

output_path = pdf_path.replace(".pdf", ".docx")
build_docx(pdf_path, all_pages_lines, all_tables, output_path)
print("Saved:", output_path)

In [ ]:
files.download(output_path)